<a href="https://colab.research.google.com/github/Shrushti-2002/AI-Interior-Designer-Budget-Aware-/blob/main/AI_Interior_Designer_(Budget_Aware).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
!pip install -q torch torchvision gradio pillow pandas numpy opencv-python

In [52]:
import torch
import torchvision.models as models
import torchvision.transforms as T
import pandas as pd
import numpy as np
import gradio as gr
from PIL import Image

In [53]:
FURNITURE_PATH = "furniture_data.csv"

df = pd.read_csv(FURNITURE_PATH)
df.head()

,item_id,name,category,style,price,color
0,1,Modern Sofa,sofa,modern,800,gray
1,2,Wood Coffee Table,table,scandinavian,300,brown
2,3,Minimalist Lamp,lamp,minimalist,150,white
3,4,Classic Armchair,chair,classic,600,beige
4,5,Modern TV Stand,storage,modern,400,black


In [54]:
resnet = models.resnet50(pretrained=True)
resnet.fc = torch.nn.Identity()
resnet.eval()

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [55]:
def extract_embedding(image: Image.Image):
    img = transform(image).unsqueeze(0)
    with torch.no_grad():
        embedding = resnet(img).numpy()[0]
    return embedding / np.linalg.norm(embedding)

In [56]:
ROOM_TYPES = [
    "bedroom",
    "living room",
    "office",
    "kitchen",
    "dining room",
    "bathroom",
    "kids room",
    "studio apartment",
    "home gym"]

def predict_room_type(embedding):
    idx = int(np.abs(embedding).sum() * 1e4) % len(ROOM_TYPES)
    return ROOM_TYPES[idx]

In [57]:
def recommend_furniture(budget, room_type):
    if room_type == "bedroom":
        required = ["bed", "table", "shelf"]
    elif room_type == "office":
        required = ["desk", "chair", "shelf"]
    else:
        required = ["sofa", "table", "shelf"]

    selected = []
    remaining = budget

    for cat in required:
        options = df[df["category"] == cat].sort_values("price")
        for _, row in options.iterrows():
            if row["price"] <= remaining:
                selected.append(row)
                remaining -= row["price"]
                break

    return selected, remaining

In [58]:
def design_room(image, budget):
    image = image.convert("RGB")

    embedding = extract_embedding(image)
    room_type = predict_room_type(embedding)

    items, remaining = recommend_furniture(budget, room_type)

    output = f"🏠 Room Type: {room_type}\n\n"
    output += "🪑 Recommended Furniture:\n"

    for item in items:
        output += f"- {item['item_id']} (${item['price']}, {item['style']})\n"

    output += f"\n💰 Remaining Budget: ${remaining}"

    return output

In [59]:
app = gr.Interface(
    fn=design_room,
    inputs=[
        gr.Image(type="pil", label="Upload Room Image"),
        gr.Number(label="Budget ($)")
    ],
    outputs=gr.Textbox(
        label="Design Recommendations",
        lines=15,
        max_lines=30
    ),
    title="AI Interior Designer (Budget-Aware)",
    description="Upload a room image and budget to get furniture & layout recommendations."
)

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8001812b37a2ba97d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Created dataset file at: .gradio/flagged/dataset1.csv


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8001812b37a2ba97d1.gradio.live
